# Week 9 - Day 1: AI Agent Fundamentals

## Learning Objectives

By the end of this notebook, you will understand:

1. What an AI Agent is
2. Difference between a chatbot and an AI Agent
3. Agent perception and decision-making
4. Tools in AI Agents
5. Tool calling
6. Agent planning
7. Agent execution
8. Observation and feedback
9. Agent loops
10. Agent memory
11. Short-term and long-term memory
12. Agent goals
13. Agent state
14. Agent autonomy
15. Single-agent architecture
16. Multi-agent architecture
17. Basic agent workflow
18. How LangChain, CrewAI, and AutoGen fit into agent development

## Week 9 Project

Build a Multi-Agent Assistant.

Target architecture:

                    User
                      ↓
                 Main Agent
                ↙    ↓    ↘
          Research   Data   Writer
           Agent    Agent   Agent
                ↘    ↓    ↙
                  Result
                    ↓
               Final Answer

# 1. What is an AI Agent?

An AI Agent is a software system that uses an AI model to:

- Understand a goal
- Decide what actions are needed
- Select appropriate tools
- Execute actions
- Observe results
- Continue reasoning or planning
- Produce a final result

A simple way to think about an agent:

Agent = AI Model + Tools + Decision Making + State + Goal

A chatbot mainly generates responses.

An agent can take actions to accomplish a task.

# Simple Example

User:

"Find the latest sales report, calculate the total sales,
and prepare a summary."

A normal chatbot may answer:

"I cannot access your sales report."

An agent can potentially:

1. Search the file system
2. Find the sales report
3. Read the file
4. Calculate total sales
5. Analyze the results
6. Write a summary
7. Return the result

The important difference is:

Chatbot:
Question → Answer

Agent:
Goal → Plan → Action → Observation → More Action → Result

# 2. Chatbot vs AI Agent

## Normal Chatbot

User
 ↓
LLM
 ↓
Answer

The LLM primarily generates a response based on:

- User input
- Conversation context
- Its learned knowledge
- Retrieved context, if RAG is used

---

## AI Agent

User
 ↓
Agent
 ↓
Understand Goal
 ↓
Plan
 ↓
Choose Tool
 ↓
Execute Tool
 ↓
Observe Result
 ↓
Decide Next Step
 ↓
Answer

An agent can interact with external systems through tools.

# Key Difference

The biggest conceptual difference is not simply:

"Agent = smarter chatbot."

Instead:

A chatbot is primarily response-oriented.

An agent is action-oriented and goal-oriented.

An agent can decide:

"What should I do next to accomplish this task?"

# 3. Real-World Example

Consider this request:

"Find the best-selling product last month and prepare a short report."

An agent might perform:

Step 1:
Understand the goal.

Step 2:
Identify that sales data is required.

Step 3:
Choose a database tool.

Step 4:
Query the database.

Step 5:
Observe the returned data.

Step 6:
Calculate the best-selling product.

Step 7:
Use a writing tool or LLM to prepare the report.

Step 8:
Return the final report.

The agent is coordinating multiple actions.

# 4. Main Components of an AI Agent

A basic AI Agent contains several components.

## 1. Goal

What should the agent accomplish?

Example:

"Analyze this sales dataset."

---

## 2. LLM

The language model provides reasoning and language capabilities.

---

## 3. Tools

External functions the agent can use.

Examples:

- Calculator
- Search
- Database
- Python
- API
- File reader
- Email system

---

## 4. Planning

Determines what actions should be performed.

---

## 5. Memory / State

Stores information needed during or across tasks.

---

## 6. Observation

The result returned after a tool is executed.

---

## 7. Action

The next step chosen by the agent.

---

## 8. Final Answer

The result returned to the user.

# 5. Basic Agent Architecture

                 USER
                   ↓
                 AGENT
                   ↓
            ┌──────┴──────┐
            ↓             ↓
          PLAN          MEMORY
            ↓
        TOOL SELECTION
            ↓
       TOOL EXECUTION
            ↓
        OBSERVATION
            ↓
       NEXT DECISION
            ↓
          ANSWER

The agent can repeat:

Plan
→ Action
→ Observation
→ Plan
→ Action
→ Observation

until the goal is completed.

# 6. Goal-Oriented AI

An agent needs a goal.

Example:

Goal:
"Calculate the average order value."

The agent identifies that it needs:

1. Order data
2. Total sales
3. Number of orders
4. Calculation

Then it can decide which tools are required.

Without a goal, tool usage can become meaningless.

The agent should select actions that move toward completing the user's task.

# 7. What is a Tool?

A tool is an external function or capability that an AI Agent can invoke.

Examples:

### Calculator

calculate(100 + 200)

### Database

query_database("SELECT * FROM employees")

### Search

search_web("latest AI research")

### Python

run_python("calculate average")

### File Reader

read_file("sales.xlsx")

### API

get_weather("Chennai")

The LLM decides when a tool is useful, while the application executes the actual function.

# Important Concept

The LLM itself does not magically execute arbitrary Python code,
access your database, or send an email.

The application provides tools.

The model can decide:

"I need to use the calculator."

The application then executes the calculator function.

Tool calling connects the model's decision to actual software actions.

In [1]:
def calculator(a, b, operation):
    if operation == "add":
        return a + b
    
    elif operation == "subtract":
        return a - b
    
    elif operation == "multiply":
        return a * b
    
    elif operation == "divide":
        return a / b
    
    else:
        return "Unsupported operation"

In [2]:
result = calculator(
    10,
    5,
    "multiply"
)

print("Result:", result)

Result: 50


# What happened?

We created a simple tool:

calculator()

The tool accepts:

- a
- b
- operation

The tool returns a result.

An AI Agent could be given this function as one of its available tools.

The model can then determine when calculation is required.

# 8. Tool Calling

Tool calling is the mechanism that allows an LLM to request
execution of a predefined tool.

Conceptual flow:

User:
"What is 25 × 40?"

        ↓

LLM:
"I should use the calculator."

        ↓

Tool Call:
calculator(25, 40, "multiply")

        ↓

Tool Result:
1000

        ↓

LLM:
"The answer is 1000."

This is different from asking the LLM to perform every action itself.

# 9. Tool Schema

For an LLM to use a tool reliably, the tool should have a clear description.

Example:

Tool name:
calculator

Description:
Performs arithmetic calculations.

Arguments:

a:
number

b:
number

operation:
add | subtract | multiply | divide

A clear schema helps the model understand:

- What the tool does
- When to use it
- What inputs it requires
- What output it produces

In [3]:
tool_definition = {
    "name": "calculator",
    "description": "Performs basic arithmetic calculations.",
    "parameters": {
        "a": "number",
        "b": "number",
        "operation": "add | subtract | multiply | divide"
    }
}

print(tool_definition)

{'name': 'calculator', 'description': 'Performs basic arithmetic calculations.', 'parameters': {'a': 'number', 'b': 'number', 'operation': 'add | subtract | multiply | divide'}}


# 10. The Agent Loop

One of the most important concepts in agent systems is the agent loop.

A simplified agent loop is:

1. Receive goal
2. Understand task
3. Plan
4. Select action
5. Call tool
6. Receive result
7. Observe result
8. Decide next action
9. Repeat if necessary
10. Produce final answer

Visual representation:

        ┌───────────────┐
        │     Goal      │
        └───────┬───────┘
                ↓
             PLAN
                ↓
          SELECT TOOL
                ↓
          EXECUTE TOOL
                ↓
           OBSERVE
                ↓
        Is task complete?
           ↙       ↘
         No        Yes
         ↓          ↓
      PLAN       ANSWER

# 11. Simulating an Agent Loop

Before using an LLM, we can understand the concept with
a simple Python simulation.

Goal:

Calculate:

25 × 40

The agent decides:

1. Calculation is required.
2. Use calculator.
3. Observe result.
4. Return answer.

In [4]:
goal = "Calculate 25 multiplied by 40"

print("Goal:", goal)

print("\nAgent decision:")
print("Use calculator tool.")

tool_result = calculator(
    25,
    40,
    "multiply"
)

print("\nTool result:", tool_result)

print("\nFinal answer:")
print(f"25 × 40 = {tool_result}")

Goal: Calculate 25 multiplied by 40

Agent decision:
Use calculator tool.

Tool result: 1000

Final answer:
25 × 40 = 1000


# 12. Agent Planning

Planning means determining the actions required to accomplish
a goal.

Example:

User:
"Analyze my sales data and create a summary."

Possible plan:

1. Locate sales file
2. Load dataset
3. Inspect columns
4. Clean missing values
5. Calculate KPIs
6. Identify trends
7. Generate summary
8. Return report

The plan can be:

- Explicit
- Implicit
- Generated dynamically

Different agent frameworks implement planning differently.

# Planning vs Execution

Planning:

"What should I do?"

Execution:

"Actually perform the action."

Example:

Planning:
"I need sales data."

Execution:
"Call read_file('sales.xlsx')."

Observation:
"File contains 10,000 orders."

Next planning step:
"I need to calculate total sales."

Execution:
"Run calculation."

The separation between decision and execution is important for agent architecture.

# 13. Multi-Step Agent Task

Consider:

"Find the total sales for Chennai and write a short summary."

Possible agent workflow:

User Request
      ↓
Understand Task
      ↓
Find sales data
      ↓
Read data
      ↓
Filter Chennai
      ↓
Calculate total sales
      ↓
Analyze result
      ↓
Write summary
      ↓
Final Answer

One user request can therefore require several actions.

# 14. Observation

Observation is the result returned after an action or tool execution.

Example:

Action:
Search database.

Observation:
"120 records were found."

The agent uses the observation to decide what to do next.

Another example:

Action:
Read Excel file.

Observation:
"The file contains columns:
OrderID, CustomerID, ProductID, Sales."

The agent now knows what information is available.

Therefore:

Action → Observation → Next Decision

# 15. Agent State

State represents information the agent currently needs to continue
working on a task.

Example:

{
    "goal": "Analyze sales",
    "file": "sales.xlsx",
    "total_orders": 5242,
    "total_sales": 347412483.5,
    "current_step": "generate_summary"
}

State can contain:

- User request
- Current plan
- Tool results
- Intermediate calculations
- Errors
- Current step
- Retrieved information
- Final result

State is especially important for multi-step workflows.

# 16. Agent Memory

Memory allows an agent system to retain information.

There are two broad concepts:

## Short-Term Memory

Information needed during the current interaction or task.

Example:

User:
"My name is Jayasri."

Later in the same conversation:

"What is my name?"

The conversation history provides the information.

---

## Long-Term Memory

Information stored beyond the immediate interaction.

Examples:

- User preferences
- Previous tasks
- Important profile information
- Historical interactions

Long-term memory usually requires explicit storage such as:

- Database
- Vector database
- Key-value store
- Memory service

# 17. Memory vs Context

These concepts are related but not identical.

## Context

Information currently provided to the model.

Example:

The current conversation messages.

## Memory

Information intentionally retained for future use.

Example:

A stored user preference.

Simple distinction:

Context:
"What does the model know right now?"

Memory:
"What information can the system retain and retrieve later?"

In [5]:
agent_memory = {}

agent_memory["user_name"] = "Jayasri"
agent_memory["preferred_language"] = "Python"

print(agent_memory)

{'user_name': 'Jayasri', 'preferred_language': 'Python'}


In [6]:
print("User name:", agent_memory["user_name"])
print("Preferred language:", agent_memory["preferred_language"])

User name: Jayasri
Preferred language: Python


This is only a simple Python dictionary.

Real agent memory systems may use:

- PostgreSQL
- Redis
- Vector databases
- Document databases
- Dedicated memory systems

The storage technology depends on the type of memory required.

# 18. Agent Autonomy

Autonomy refers to how independently an agent can decide and
execute actions.

Low autonomy:

User
↓
Agent
↓
One predefined tool
↓
Answer

Higher autonomy:

User
↓
Agent
↓
Plan
↓
Choose among multiple tools
↓
Execute
↓
Observe
↓
Re-plan
↓
Execute more actions
↓
Answer

Higher autonomy also means higher risk.

Therefore, real systems should use:

- Tool permissions
- Input validation
- Access controls
- Execution limits
- Human approval where appropriate
- Logging
- Monitoring

# 19. Agent Safety

An agent that can take actions can potentially cause real-world effects.

Examples:

- Sending an email
- Updating a database
- Deleting a file
- Making a purchase
- Changing account settings

Therefore, tool access should be controlled.

A useful principle:

Read-only actions are generally easier to control than
actions that modify external systems.

For sensitive actions, an application may require:

User approval
↓
Tool execution

instead of:

Agent decision
↓
Immediate execution

# 20. Single-Agent Architecture

A single agent can have multiple tools.

Example:

                  USER
                    ↓
                  AGENT
                    ↓
        ┌───────────┼───────────┐
        ↓           ↓           ↓
    Calculator   Database    Search
        │           │           │
        └───────────┼───────────┘
                    ↓
                 RESULT
                    ↓
                FINAL ANSWER

The agent decides which tool is appropriate.

# 21. Multi-Agent Systems

A multi-agent system contains multiple specialized agents.

Each agent can have a specific responsibility.

Example:

                    USER
                      ↓
                MAIN AGENT
               ↙    ↓    ↘
          RESEARCH  DATA  WRITER
           AGENT   AGENT   AGENT
               ↘    ↓    ↙
                  RESULT
                    ↓
               FINAL ANSWER

Instead of one agent performing everything,
specialized agents divide the work.

# 22. Specialized Agents

### Research Agent

Responsibilities:

- Search information
- Collect sources
- Summarize findings

### Data Agent

Responsibilities:

- Read datasets
- Perform calculations
- Analyze data
- Generate statistics

### Writer Agent

Responsibilities:

- Organize findings
- Write reports
- Improve clarity
- Produce final output

### Main Agent / Coordinator

Responsibilities:

- Understand the user's goal
- Delegate tasks
- Combine results
- Produce final response

# 23. Why Use Multiple Agents?

Multi-agent systems can be useful when a task contains
distinct areas of expertise.

Example:

"Research competitors, analyze our sales data, and write
an executive report."

This contains three different tasks:

Research
+
Data Analysis
+
Writing

Specialized agents can handle each responsibility.

However:

Multi-agent systems also add complexity.

More agents can mean:

- More latency
- More API calls
- More cost
- More coordination
- More failure points

Therefore, multiple agents should be used when specialization
actually provides value.

# 24. Agent Frameworks

Week 9 will introduce several frameworks.

## LangChain

A framework/ecosystem for building applications around
LLMs, tools, retrieval, agents, and workflows.

---

## CrewAI

A framework focused on role-based agent collaboration
and multi-agent workflows.

Example:

Researcher
+
Analyst
+
Writer

---

## AutoGen

A framework for building applications involving
LLM agents and agent-to-agent interactions.

---

We will study these frameworks later in the week.

For now, focus on the underlying concepts.

Frameworks change.

Agent architecture concepts are more fundamental.

# 25. RAG vs AI Agent

RAG:

User
↓
Retrieve relevant information
↓
LLM
↓
Answer

Agent:

User
↓
Agent
↓
Decide what to do
↓
Use tools
↓
Observe results
↓
Continue
↓
Answer

RAG can also be a tool inside an agent.

Example:

Agent
↓
Needs company information
↓
Calls RAG search tool
↓
Receives relevant documents
↓
Continues reasoning

# 26. Agent + RAG

An agent can use a RAG system as one of its tools.

Example:

User:
"What is our remote work policy and how does it compare
with our competitors?"

Agent:

Step 1:
Use internal RAG system.

Step 2:
Retrieve company remote-work policy.

Step 3:
Use web search tool.

Step 4:
Research competitors.

Step 5:
Compare the information.

Step 6:
Ask writer component to create a summary.

Step 7:
Return final answer.

This demonstrates how RAG and agents can work together.

In [8]:
def choose_tool(task):
    task = task.lower()

    if "calculate" in task or "multiply" in task or "add" in task:
        return "calculator"

    elif "search" in task or "research" in task:
        return "search"

    elif "database" in task or "employee" in task:
        return "database"

    else:
        return "llm_only"


tasks = [
    "Calculate 100 multiplied by 5",
    "Research the latest AI trends",
    "Find employee information",
    "Explain what machine learning is"
]

for task in tasks:
    print(task, "→", choose_tool(task))

Calculate 100 multiplied by 5 → calculator
Research the latest AI trends → search
Find employee information → database
Explain what machine learning is → llm_only


# 27. What Did We Just Build?

This is NOT a full AI Agent.

It is a simplified rule-based simulation of tool selection.

Real LLM-based agents can use an LLM to determine:

- Which tool to use
- What arguments to provide
- Whether another tool is needed
- Whether the task is complete

The conceptual flow remains similar:

Task
↓
Decision
↓
Tool
↓
Result
↓
Next Decision

In [9]:
agent_state = {
    "goal": "Calculate total sales",
    "current_step": "retrieve_data",
    "tools_used": [],
    "observations": [],
    "result": None
}

print(agent_state)

{'goal': 'Calculate total sales', 'current_step': 'retrieve_data', 'tools_used': [], 'observations': [], 'result': None}


In [10]:
agent_state["tools_used"].append("database")

agent_state["observations"].append(
    "Retrieved 5242 sales records."
)

agent_state["current_step"] = "calculate_total"

print(agent_state)

{'goal': 'Calculate total sales', 'current_step': 'calculate_total', 'tools_used': ['database'], 'observations': ['Retrieved 5242 sales records.'], 'result': None}


In [11]:
def simple_agent(goal):
    state = {
        "goal": goal,
        "steps": []
    }

    print("Goal:", goal)

    # Planning
    state["steps"].append("Understand goal")
    print("\n1. Understand goal")

    # Tool selection
    state["steps"].append("Select tool")
    print("2. Select appropriate tool")

    # Execution
    state["steps"].append("Execute tool")
    print("3. Execute tool")

    # Observation
    state["steps"].append("Observe result")
    print("4. Observe result")

    # Final response
    state["steps"].append("Generate final answer")
    print("5. Generate final answer")

    return state


state = simple_agent(
    "Calculate the total sales"
)

print("\nAgent state:")
print(state)

Goal: Calculate the total sales

1. Understand goal
2. Select appropriate tool
3. Execute tool
4. Observe result
5. Generate final answer

Agent state:
{'goal': 'Calculate the total sales', 'steps': ['Understand goal', 'Select tool', 'Execute tool', 'Observe result', 'Generate final answer']}


# 28. Agent Workflow

The basic agent workflow is:

USER
 ↓
GOAL
 ↓
UNDERSTAND
 ↓
PLAN
 ↓
SELECT TOOL
 ↓
EXECUTE
 ↓
OBSERVE
 ↓
RE-PLAN IF NECESSARY
 ↓
COMPLETE TASK
 ↓
FINAL ANSWER

Remember:

An agent is not simply an LLM with a fancy name.

The important capability is the controlled loop between:

Decision
→ Action
→ Observation
→ Decision

# 29. Exercise 1 - Identify Agent Components

For the following task:

"Read my sales CSV, calculate monthly sales, and create a summary."

Identify:

1. Goal
2. Required tools
3. Possible plan
4. Expected observations
5. Final output

Write your answer in Python or Markdown.

# Exercise 2 - Tool Selection

For each request, identify the most appropriate tool.

1. "Calculate 250 * 30"
2. "Find employee E102 information"
3. "Search for recent AI research"
4. "Read the contents of sales.xlsx"
5. "Write a summary of these findings"

Possible tools:

- Calculator
- Database
- Search
- File Reader
- Writer / LLM

# Exercise 3 - Design an Agent Loop

Design an agent workflow for:

"Find the total number of orders in Chennai and prepare a short report."

Your workflow should contain:

User Goal
↓
Plan
↓
Tool Selection
↓
Execution
↓
Observation
↓
Next Step
↓
Final Answer

# Exercise 4 - Single Agent vs Multi-Agent

Decide whether each task could reasonably use a single agent
or multiple specialized agents.

### Task A
"Calculate the average of five numbers."

### Task B
"Research 10 companies, analyze their financial data,
and write a detailed comparison report."

### Task C
"Find an employee's department from a database."

Explain your reasoning.

Do not assume multi-agent is automatically better.
More components mean more opportunities for things to break.

# 30. AI Agent Interview Questions

### Q1. What is an AI Agent?

An AI Agent is a system that uses an AI model to understand a goal,
make decisions, use tools, observe results, and take additional
actions when required to complete a task.

---

### Q2. What is the difference between a chatbot and an AI Agent?

A chatbot primarily generates responses.

An agent can plan and execute actions using tools to accomplish
a goal.

---

### Q3. What is tool calling?

Tool calling is the mechanism through which an LLM requests
execution of a predefined external function.

---

### Q4. Why do agents need tools?

Tools allow agents to interact with external systems and perform
actions that the LLM cannot perform by itself.

Examples include databases, APIs, calculators, search systems,
and file readers.

---

### Q5. What is an agent loop?

An agent loop is the repeated process of:

Plan
→ Action
→ Observation
→ Next Decision

until the task is completed.

---

### Q6. What is agent planning?

Planning is determining which actions should be performed to
achieve a user's goal.

---

### Q7. What is agent state?

Agent state is the information required to track the current
progress and continue a task.

---

### Q8. What is agent memory?

Agent memory is stored information that can be used during or
across interactions.

---

### Q9. What is the difference between short-term and long-term memory?

Short-term memory stores information needed for the current task
or conversation.

Long-term memory persists information for future interactions.

---

### Q10. What is a multi-agent system?

A multi-agent system contains multiple specialized agents that
coordinate to complete a larger task.

---

### Q11. Why use multiple agents?

Multiple agents can divide a complex task into specialized
responsibilities.

For example:

Research Agent
+
Data Agent
+
Writer Agent

---

### Q12. What are disadvantages of multi-agent systems?

They can introduce:

- Additional latency
- More API calls
- Higher cost
- Coordination complexity
- More failure points

---

### Q13. Can RAG be used by an AI Agent?

Yes.

A RAG retrieval system can be exposed as a tool that an agent
can call when it needs information from documents.

---

### Q14. What is the role of an LLM in an agent?

The LLM can interpret the task, determine actions, select tools,
generate tool arguments, interpret results, and generate the final
response.

The actual tool execution is handled by the application.

---

### Q15. What are LangChain, CrewAI, and AutoGen?

They are frameworks/ecosystems used to build LLM-based
applications and agent workflows.

Their exact APIs may change, so understanding the underlying
agent concepts is more important than memorizing framework syntax.

# 31. Day 1 AI Agent Cheat Sheet

## AI Agent

AI system that can pursue a goal using reasoning/decision-making,
tools, state, and actions.

## Goal

What the agent is trying to accomplish.

## LLM

Provides language understanding and decision-making capabilities.

## Tool

External function or capability available to the agent.

Examples:

- Search
- Database
- Calculator
- API
- Python
- File reader

## Tool Calling

LLM requests execution of a predefined tool.

## Planning

Determining what actions are needed.

## Action

An operation performed by the agent or a tool.

## Observation

The result returned after an action.

## State

Information representing the current progress of the task.

## Memory

Information retained for current or future use.

## Agent Loop

Goal
→ Plan
→ Action
→ Observation
→ Re-plan
→ Answer

## Single Agent

One agent with potentially multiple tools.

## Multi-Agent

Multiple specialized agents collaborating on a task.

## RAG + Agent

RAG can be used as one of the agent's tools.

## Frameworks

LangChain
CrewAI
AutoGen

# 33. Day 1 Final Summary

Today we learned the foundation of AI Agents.

## Normal Chatbot

User
↓
LLM
↓
Answer

## AI Agent

User
↓
Agent
↓
Understand Goal
↓
Plan
↓
Choose Tool
↓
Execute Tool
↓
Observe Result
↓
Re-plan if necessary
↓
Final Answer

## Multi-Agent

                    USER
                      ↓
                MAIN AGENT
               ↙    ↓    ↘
          RESEARCH  DATA  WRITER
           AGENT   AGENT   AGENT
               ↘    ↓    ↙
                  RESULT
                    ↓
               FINAL ANSWER

## Core Formula

AI Agent =
LLM
+
Goal
+
Tools
+
Planning
+
State
+
Execution
+
Observation

## Agent Loop

PLAN
 ↓
ACT
 ↓
OBSERVE
 ↓
RE-PLAN
 ↓
ACT
 ↓
OBSERVE
 ↓
ANSWER

## Important Principle

An AI Agent is not simply a chatbot with a different name.

The important capability is the ability to use tools and
take controlled actions toward a goal.

## Week 9 Project

Multi-Agent Assistant

Main Agent
├── Research Agent
├── Data Agent
└── Writer Agent

The agents will eventually coordinate their work to produce
a final response.